# Metatlas2 Standalone Development Workflow

Execute cells sequentially to run the complete metatlas2 pipeline using production code.

In [ ]:
import os
import logging
from pathlib import Path

# Import workflow modules
import metatlas2.workflows as wfs
import metatlas2.load_tools as ldt
import metatlas2.logging_config as lcf
from metatlas2.add_compounds_to_db import add_compounds_to_db
from metatlas2.add_atlases_to_db import add_atlases_to_db
from metatlas2.run_targeted_analysis import set_up_paths

# Set up logging
lcf.setup_logging(log_level=logging.INFO, log_to_stdout=True)
logger = lcf.get_logger("standalone_workflow")
logger.info("Standalone development workflow starting...")

In [ ]:
DATA_DIR = Path(os.environ.get('METATLAS_DATA_DIR', Path.home() / '.metatlas2-dev'))
os.environ['METATLAS_DATA_DIR'] = str(DATA_DIR)

In [ ]:
project_name = "20260101_JGI_XX_000000_STANDALONE-DEV_test_EXP000_HILICZ_TESTXXXX"
rt_alignment_number = 0
analysis_number = 0
compounds_config = DATA_DIR / "configs" / "compounds_config.yaml"
atlases_config = DATA_DIR / "configs" / "atlases_config.yaml"
analysis_config = DATA_DIR / "configs" / "analysis_config.yaml"

In [ ]:
logger.info(f"Loading configs...")
config = ldt.load_metatlas2_config(str(analysis_config))
logger.info("Configuration files loaded successfully")

In [ ]:
logger.info("Adding compounds to database...")
add_compounds_to_db(config_path=str(compounds_config), overwrite_db=True)
logger.info("Compounds added successfully")

In [ ]:
logger.info("Creating atlases...")
add_atlases_to_db(config_path=str(atlases_config))
logger.info("Atlases created successfully.")

In [ ]:
logger.info("*Before proceeding, update the atlas UIDs from the output in the previous cell*")
qc_atlas_uid = ""
ema_atlas_uid = ""

In [ ]:
logger.info("Setting up project paths...")
config['RT_ALIGNMENT']['HILICZ']['ATLAS']['uid'] = qc_atlas_uid
config['TARGETED_ANALYSES']['HILICZ']['POS']['EMA']['ATLAS']['uid'] = ema_atlas_uid
paths = set_up_paths(
    config=config,
    project_name=project_name,
    rt_alignment_number=rt_alignment_number,
    analysis_number=analysis_number,
)

logger.info(f"Project database: {paths['project_db_path']}")
logger.info(f"Analysis output: {paths['analysis_output_dir']}")

In [ ]:
logger.info("Running project setup...")
wfs.run_project_setup(
    project_name=project_name,
    config=config,
    paths=paths,
    overwrite_existing=True,
)
logger.info("Project setup complete")

In [ ]:
logger.info("Running RT alignment...")
wfs.run_rt_alignment(
    project_name=project_name,
    rt_alignment_number=rt_alignment_number,
    config=config,
    paths=paths,
)
logger.info("RT alignment complete")

In [ ]:
logger.info("Running auto-identification...")
nb_paths = wfs.run_auto_identification(
    project_name=project_name,
    config=config,
    paths=paths,
    rt_alignment_number=rt_alignment_number,
    analysis_number=analysis_number,
    config_path=str(analysis_config),
    image_tag="latest",
)
logger.info(f"Auto-identification complete. GUI notebooks created at: {nb_paths}")